In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", 
    "langchain", "langchain-community",
    "chromadb", "pypdf", "sentence-transformers", "python-dotenv",
    "streamlit", "ipykernel", "langchain-text-splitters",
    "langchain-google-genai"
])


from dotenv import load_dotenv
import os

load_dotenv()
key = os.getenv("GOOGLE_API_KEY")
print("Key found:", key)

Key found: Google_API_Key

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(r"C:\Users\rudre\OneDrive\Desktop\rag-qa-project\data\attention.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0].page_content[:300])

c:\Users\rudre\OneDrive\Desktop\rag-qa-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 15 pages
Split into 52 chunks

Example chunk:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [3]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db_path = r"C:\Users\rudre\OneDrive\Desktop\rag-qa-project\outputs\chroma_db"

if os.path.exists(db_path) and os.listdir(db_path):
    # Reload existing vectorstore from disk
    vectorstore = Chroma(
        persist_directory=db_path,
        embedding_function=embeddings
    )
    print(f"Reloaded vectorstore — {vectorstore._collection.count()} chunks found")
else:
    # Create new vectorstore
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_path
    )
    print(f"Created new vectorstore — {vectorstore._collection.count()} chunks stored")

C:\Users\rudre\AppData\Local\Temp\ipykernel_18416\1955043121.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3472.11it/s]
C:\Users\rudre\AppData\Local\Temp\ipykernel_18416\1955043121.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Reloaded vectorstore — 104 chunks found


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# ==============================
# SETUP GEMINI MODEL
# ==============================
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)

# ==============================
# CREATE RETRIEVER
# ==============================
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# ==============================
# CREATE PROMPT TEMPLATE
# ==============================
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question ONLY from the provided context.

If the answer is not present in the context,
say:
"I don't have enough information to answer this."

Context:
{context}

Question:
{question}

Answer:
""")

# ==============================
# FORMAT RETRIEVED DOCUMENTS
# ==============================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ==============================
# BUILD RAG CHAIN
# ==============================
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ==============================
# ASK QUESTION
# ==============================
question = "What is the main idea of attention mechanism?"

answer = chain.invoke(question)

# ==============================
# PRINT OUTPUT
# ==============================
print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
What is the main idea of attention mechanism?

ANSWER:
Based on the provided context, the main idea of an attention function is mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum.
